In [1]:
import matplotlib.pyplot as plt
import numpy as np
import qutip as qt
from IPython.display import display, Math, Latex

from IPython.display import display, Math, Latex
import ipywidgets as widgets
from matplotlib.widgets import Slider, Button

from scipy.fft import fft, ifft
import scipy.special as sp
import scipy.optimize 

In [2]:
### Phase preserving, so assume all coefficients are Real

In [3]:
Delta = 0 # w0-wp
kappa = 300e6 * 2*np.pi
drive = 0.2
n_levels = 150
step = 30

lam_range = -np.linspace(0.0,1.94,step)*kappa/4

In [4]:
def Diff_DPA(Delta, kappa, lam, drive, n_levels):
    
    a = qt.destroy(n_levels)
    
    H_DPA = Delta * a.dag() * a + lam/2 * (a.dag() * a.dag() + a * a)
    
    epsilon = drive*kappa
    H_probe = epsilon * (a.dag() + a)
  
    c_op_list = [np.sqrt(kappa) * a]
    final_state = qt.steadystate(H_DPA+H_probe, c_op_list)

    M = qt.expect(a*a, final_state) - qt.expect(a, final_state)**2
    N = qt.expect(a.dag()*a, final_state) - np.abs(qt.expect(a, final_state))**2
    
    Diff_DPA = 1 - np.abs(M)/np.sqrt(N*(N+1/2))

    
    return Diff_DPA

In [5]:
# JPA parameters
F_JPA = np.pi/4
EJ = 2000e9 * 2*np.pi
EC_J = 9e6 * 2*np.pi
phi_zpf_JPA = (2*EC_J/EJ)**0.25
JPA_ratio = phi_zpf_JPA**2/6 # |LAM/lam| ratio in abs value

Kerr = -EC_J * np.cos(F_JPA) / 2
print(Kerr/kappa)
print(JPA_ratio)

-0.010606601717798213
0.0005


In [6]:
def Diff_JPA(Delta, kappa, lam, Kerr, drive, n_levels):
    
    a = qt.destroy(n_levels)
    
    H_DPA = Delta * a.dag() * a + lam/2 * (a.dag() * a.dag() + a * a)

    LAM = - JPA_ratio * lam
    H_alpha = Kerr * a.dag() * a.dag() * a * a  + LAM *(a.dag()*a.dag()*a.dag()*a + a.dag() * a * a * a)
    
    epsilon = drive*kappa
    H_probe = epsilon * (a.dag() + a)
  
    c_op_list = [np.sqrt(kappa) * a]
    final_state = qt.steadystate(H_DPA+H_alpha+H_probe, c_op_list)

    M = qt.expect(a*a, final_state) - qt.expect(a, final_state)**2
    N = qt.expect(a.dag()*a, final_state) - np.abs(qt.expect(a, final_state))**2
    
    Diff_JPA = 1 - np.abs(M)/np.sqrt(N*(N+1/2))

    
    return Diff_JPA

In [9]:
# JPA
Dev_JPA=np.zeros(step)
Dev_KFJPA=np.zeros(step)
k = 0
for lam in lam_range:
    
    g_Q = Diff_JPA(Delta, kappa, lam, Kerr, drive, n_levels)
    Dev_JPA[k] = g_Q

    h_Q = Diff_JPA(Delta, kappa, lam, 0, drive, n_levels)
    Dev_KFJPA[k] = h_Q
    
    k = k+1
    

C:\Users\kagan\AppData\Local\Temp\ipykernel_28760\1770780636.py:19: RuntimeWarning: invalid value encountered in sqrt
  Diff_JPA = 1 - np.abs(M)/np.sqrt(N*(N+1/2))


In [7]:
x_range = -JPA_ratio*lam_range/kappa

In [8]:
x_range

array([0.00000000e+00, 8.36206897e-06, 1.67241379e-05, 2.50862069e-05,
       3.34482759e-05, 4.18103448e-05, 5.01724138e-05, 5.85344828e-05,
       6.68965517e-05, 7.52586207e-05, 8.36206897e-05, 9.19827586e-05,
       1.00344828e-04, 1.08706897e-04, 1.17068966e-04, 1.25431034e-04,
       1.33793103e-04, 1.42155172e-04, 1.50517241e-04, 1.58879310e-04,
       1.67241379e-04, 1.75603448e-04, 1.83965517e-04, 1.92327586e-04,
       2.00689655e-04, 2.09051724e-04, 2.17413793e-04, 2.25775862e-04,
       2.34137931e-04, 2.42500000e-04])

In [9]:
plt.figure(figsize=(16,10))
plt.rcParams['text.usetex'] = True
#plt.plot(-lam_range, Dev_DPA, '--', c='blue', linewidth=5, label = 'DPA')
plt.plot(x_range, Dev_JPA, '-', c='red', linewidth=10, label = r'$|K|=10^{-2}\kappa$')
plt.plot(x_range, Dev_KFJPA,'-.', c='purple', linewidth=10, label = r'$K=0$')





plt.legend(loc='upper left', fontsize=45)
plt.xlabel(r'$|\Lambda|$ / $\kappa$',fontsize=80)
plt.ylabel(r'$\Xi_{\rm JPA}$', fontsize=80)
plt.xscale('log')
plt.yscale('log')
plt.xticks(fontsize=50)
plt.yticks(fontsize=50)
plt.tight_layout()
#plt.savefig("Deviation from DPA.jpeg")

NameError: name 'Dev_JPA' is not defined

<Figure size 1600x1000 with 0 Axes>

In [13]:
import pandas as pd

In [16]:
# Sample DataFrames for demonstration
data1 = {'x_val': x_range, 'y_val_plot1': Dev_JPA}
df1 = pd.DataFrame(data1)

data2 = {'x_val': x_range, 'y_val_plot2': Dev_KFJPA}
df2 = pd.DataFrame(data2)

# Option 1: Concatenate if columns are different or you want to keep them separate
combined_df_option1 = pd.concat([df1, df2], axis=1)

In [17]:
# Exporting the first option (side-by-side columns)
combined_df_option1.to_csv('Deviation_vs_LAM.csv', index=False)
